# Chạy Pipeline Student/Advisor Link Prediction trên Kaggle

Notebook clone repository `HongHa-1606/social-link-prediction` vào `/kaggle/working`, dùng pipeline/model sẵn có, thu thập quan hệ học trò và cố vấn, tiền xử lý graph rồi gọi trainer của repo. Model dùng `core.ai.gnn_architecture.LinkPredictionModel`; kiến trúc và vòng train không được viết lại trong notebook.

Chỉ lấy hai quan hệ có hướng riêng: `student_of` (P1066) và `advisor_of` (P184). Chúng được gộp trong một truy vấn SPARQL `UNION`, nhưng vẫn giữ nhãn riêng. Mỗi nhánh tối đa 1.250 kết quả, tổng tối đa 2.500; kết quả UNION vẫn phân trang 1.000 dòng/request để tránh timeout. Test mode dùng năm sinh 1980–2000. Trên Kaggle bật **Internet** và chọn **GPU**. Push các thay đổi mới của `config/queries.py`, `infrastructure/pipelines/extractor.py`, `requirements.txt` và `scripts/train_model.py` lên GitHub trước khi clone.

Dữ liệu thật có thể ít hơn giới hạn do Wikidata không có đủ bản ghi trong khoảng năm đã chọn. Notebook đếm số dòng sau thu thập và kiểm tra graph/PyG trước khi train.

## 1. Đọc File README
Đọc README ngay trong workspace để làm nguồn kiểm tra cấu hình dữ liệu và kiến trúc model.

In [ ]:
from pathlib import Path
import sys
import re
import json
import time
import warnings
import subprocess
import importlib.util

GITHUB_REPO_URL = "https://github.com/HongHa-1606/social-link-prediction.git"
KAGGLE_WORKING = Path("/kaggle/working")
IS_KAGGLE = KAGGLE_WORKING.exists()

if IS_KAGGLE:
    PROJECT_ROOT = KAGGLE_WORKING / "social-link-prediction"
    if not (PROJECT_ROOT / "README.md").is_file():
        subprocess.run(
            ["git", "clone", "--depth", "1", GITHUB_REPO_URL, str(PROJECT_ROOT)],
            check=True,
        )
else:
    candidates = [Path.cwd(), *Path.cwd().parents]
    PROJECT_ROOT = next((path for path in candidates if (path / "README.md").is_file()), None)
    if PROJECT_ROOT is None:
        raise FileNotFoundError("Không tìm thấy README.md hoặc repo GitHub để khởi tạo project.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Kaggle đã có sẵn PyTorch; chỉ cài các thư viện dự án còn thiếu, tránh thay Torch/CUDA.
if IS_KAGGLE:
    module_to_package = {
        "pandas": "pandas",
        "numpy": "numpy",
        "igraph": "igraph",
        "torch_geometric": "torch-geometric",
        "SPARQLWrapper": "SPARQLWrapper",
        "unidecode": "Unidecode",
        "pyarrow": "pyarrow",
        "sentence_transformers": "sentence-transformers",
        "sklearn": "scikit-learn",
        "optuna": "optuna",
    }
    missing_packages = [
        package for module, package in module_to_package.items()
        if importlib.util.find_spec(module) is None
    ]
    if missing_packages:
        if importlib.util.find_spec("torch") is None:
            raise RuntimeError("Kaggle kernel không có PyTorch; chọn image/kernel có PyTorch trước.")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", *missing_packages],
            check=True,
        )

README_PATH = PROJECT_ROOT / "README.md"
readme_text = README_PATH.read_text(encoding="utf-8")
print(f"Project: {PROJECT_ROOT}")
print(f"Kaggle: {IS_KAGGLE}")
print(f"README: {README_PATH} ({len(readme_text):,} ký tự)")
for phrase in ("Wikidata", "432-dimensional", "GraphSAGE", "paraphrase-multilingual-MiniLM-L12-v2"):
    print(f"- {'Có' if phrase.lower() in readme_text.lower() else 'Không có'}: {phrase}")

In [ ]:
# Kaggle image thường có sẵn các package này; bổ sung nếu kernel tối giản.
if IS_KAGGLE:
    support_modules = {"networkx": "networkx", "tqdm": "tqdm"}
    missing_support = [
        package for module, package in support_modules.items()
        if importlib.util.find_spec(module) is None
    ]
    if missing_support:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", *missing_support],
            check=True,
        )

# LinkNeighborLoader cần pyg-lib hoặc torch-sparse tương thích với bản Torch/CUDA của Kaggle.
if IS_KAGGLE and importlib.util.find_spec("pyg_lib") is None and importlib.util.find_spec("torch_sparse") is None:
    import torch

    torch_version = torch.__version__.split("+")[0]
    cuda_version = torch.version.cuda
    cuda_tag = f"cu{cuda_version.replace('.', '')}" if cuda_version else "cpu"
    pyg_wheel_index = f"https://data.pyg.org/whl/torch-{torch_version}+{cuda_tag}.html"
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyg_lib", "torch_sparse", "-f", pyg_wheel_index],
        check=True,
    )

## 2. Phân Tích Yêu Cầu Model
README yêu cầu graph dị thể từ các triple Wikidata, nhúng văn bản 384 chiều bằng Sentence-BERT đa ngôn ngữ, năm sinh + cờ thiếu, PageRank theo quan hệ + degree; encoder GraphSAGE và decoder dùng `[z_src, z_dst, z_src * z_dst]`.

In [ ]:
from config.queries import ALL_QUERIES
from config.settings import (
    RAW_JSON_DIR,
    RAW_PARQUET_DIR,
    CLEAN_DATA_DIR,
    PYG_DATA_PATH,
    TRAINING_HISTORY_PATH,
    OUTPUT_DIM,
)

EXPECTED_QUERY_NAMES = {"student", "advisor"}
if set(ALL_QUERIES) != EXPECTED_QUERY_NAMES:
    raise ValueError(f"Cần chỉ bật student/advisor trong config/queries.py; hiện có: {sorted(ALL_QUERIES)}")

ACTIVE_RELATIONS = list(ALL_QUERIES)
RELATION_LABEL_BY_QUERY = {
    "student": "student_of",
    "advisor": "advisor_of",
}
ACTIVE_RELATION_LABELS = sorted(set(RELATION_LABEL_BY_QUERY.values()))
TEXT_EMBEDDING_DIM = 384
EXPECTED_FEATURE_DIM = TEXT_EMBEDDING_DIM + 2 + len(ACTIVE_RELATION_LABELS) + 1

print("Queries đang bật:", ACTIVE_RELATIONS)
print("Relation labels riêng model học:", ACTIVE_RELATION_LABELS)
print("Feature width dự kiến:", EXPECTED_FEATURE_DIM)
print("Model dùng: core.ai.gnn_architecture.LinkPredictionModel (GraphSAGE + shared InteractionMLP)")
print("Training gọi scripts.train_model.run_optimization()")

In [ ]:
# Smoke test giữ nguyên hai relation có hướng khác nhau.
import importlib.util

required_modules = [
    "pandas", "numpy", "igraph", "torch", "torch_geometric",
    "SPARQLWrapper", "unidecode", "pyarrow", "sentence_transformers",
    "sklearn", "optuna",
]
missing_modules = [name for name in required_modules if importlib.util.find_spec(name) is None]
if missing_modules:
    raise RuntimeError(
        f"Thiếu dependency: {missing_modules}. Cài môi trường dự án trước bằng "
        "`python -m pip install -r requirements.txt`, sau đó chọn lại kernel .venv."
    )

import pandas as pd
from infrastructure.pipelines import GraphTransformer

smoke_nodes = pd.DataFrame([
    {"id": "Q1", "name": "Student A", "type": "human", "birth_year": 1980},
    {"id": "Q2", "name": "Advisor B", "type": "human", "birth_year": 1950},
    {"id": "Q3", "name": "Student C", "type": "human", "birth_year": 1945},
])
smoke_edges = pd.DataFrame([
    {"person": "Q1", "object": "Q2", "relationship_label": "student_of"},
    {"person": "Q2", "object": "Q3", "relationship_label": "advisor_of"},
])
smoke_graph = GraphTransformer().build_graph(smoke_edges, smoke_nodes)
assert smoke_graph.vcount() == len(smoke_nodes)
assert smoke_graph.ecount() == len(smoke_edges)
assert set(smoke_graph.es["relationship_label"]) == {"student_of", "advisor_of"}
print(f"Directed relation smoke test PASS: {smoke_graph.vcount()} nodes, {smoke_graph.ecount()} edges")

## 3. Thu Thập Dữ Liệu Theo Yêu Cầu Model
Hai pattern P1066 (`student_of`) và P184 (`advisor_of`) nằm trong cùng truy vấn SPARQL `UNION`; nhãn và hướng cạnh được giữ riêng. Mỗi nhánh giới hạn 1.250 kết quả, tổng tối đa 2.500. Extractor nhận kết quả UNION theo trang 1.000 để tránh `504`; đây là pagination theo lô, không phải query riêng cho từng record.

In [ ]:
from infrastructure.pipelines import WikidataExtractor
from config.queries import STUDENT_ADVISOR_UNION_QUERY

FULL_START_YEAR = 1800
FULL_END_YEAR = 2025
TEST_MODE = True
TEST_START_YEAR = 1980
TEST_END_YEAR = 2000
MAX_RECORDS_TOTAL = 2500
RUN_EXTRACTION = True
CONTACT_EMAIL = "your-email@example.com"  # thay bằng email liên hệ của bạn

if TEST_MODE:
    START_YEAR, END_YEAR = TEST_START_YEAR, TEST_END_YEAR
else:
    START_YEAR, END_YEAR = FULL_START_YEAR, FULL_END_YEAR

# One date interval keeps the 1,250 cap scoped to each UNION branch over the whole selected range.
YEAR_STEP = END_YEAR - START_YEAR + 1
if MAX_RECORDS_TOTAL % 2 != 0:
    raise ValueError("MAX_RECORDS_TOTAL phải chia hết cho hai nhánh UNION.")
PER_RELATION_LIMIT = MAX_RECORDS_TOTAL // 2
combined_query = STUDENT_ADVISOR_UNION_QUERY.replace(
    "##PER_RELATION_LIMIT##",
    str(PER_RELATION_LIMIT),
)

if not CONTACT_EMAIL or CONTACT_EMAIL == "your-email@example.com":
    raise ValueError("Hãy thay CONTACT_EMAIL bằng email liên hệ trước khi gọi Wikidata.")
USER_AGENT = f"SocialLinkPredictionNotebook/1.0 ({CONTACT_EMAIL})"

print("Một SPARQL UNION, hai nhãn riêng:", sorted(RELATION_LABEL_BY_QUERY.values()))
print(f"Tổng cap={MAX_RECORDS_TOTAL}; cap mỗi nhánh={PER_RELATION_LIMIT}; năm={START_YEAR}-{END_YEAR}")

if RUN_EXTRACTION:
    extractor = WikidataExtractor(user_agent=USER_AGENT)
    extracted_count = extractor.fetch_combined_relationships(
        combined_query=combined_query,
        name="student_advisor",
        start=START_YEAR,
        end=END_YEAR,
        step=YEAR_STEP,
        page_size=1000,
        max_results=MAX_RECORDS_TOTAL,
    )
    print(f"Tổng số record đã lấy: {extracted_count}")
else:
    print("Đã bỏ qua truy vấn mạng. Đổi RUN_EXTRACTION=True để bắt đầu thu thập.")

In [ ]:
import pandas as pd
import numpy as np


def binding_value(binding, key, default=None):
    value = binding.get(key)
    return value.get("value", default) if isinstance(value, dict) else default


def normalize_qid(value):
    if value is None:
        return None
    candidate = str(value).rsplit("/", 1)[-1]
    return candidate if re.fullmatch(r"Q[1-9][0-9]*", candidate) else None


raw_file = RAW_JSON_DIR / "raw_data_student_advisor.json"
if not raw_file.is_file():
    raise FileNotFoundError(
        f"Không tìm thấy combined raw file: {raw_file}. Hãy chạy cell UNION thu thập trước."
    )
try:
    payload = json.loads(raw_file.read_text(encoding="utf-8"))
except (OSError, json.JSONDecodeError) as error:
    raise ValueError(f"Không đọc được combined raw JSON {raw_file}: {error}") from error

EDUCATION_RELATION_LABELS = {"student_of", "advisor_of"}
records_by_relation = {relation: 0 for relation in EDUCATION_RELATION_LABELS}
edge_rows = []
for binding in payload.get("results", {}).get("bindings", []):
    person_id = normalize_qid(binding_value(binding, "person"))
    object_id = normalize_qid(binding_value(binding, "object"))
    relation = binding_value(binding, "relationshipLabel")
    if relation not in EDUCATION_RELATION_LABELS:
        raise ValueError(f"UNION trả về relation không mong muốn: {relation!r}")
    if not person_id or not object_id:
        continue

    edge_rows.append({
        "person": person_id,
        "object": object_id,
        "relationship_label": relation,
        "person_label": binding_value(binding, "personLabel"),
        "person_description": binding_value(binding, "personDescription"),
        "birth_year": binding_value(binding, "birthYear"),
        "birth_place": binding_value(binding, "birthPlaceLabel"),
        "country": binding_value(binding, "countryLabel"),
        "object_label": binding_value(binding, "objectLabel"),
        "object_description": binding_value(binding, "objectDescription"),
    })
    records_by_relation[relation] += 1

if not edge_rows:
    raise ValueError("UNION không trả về cạnh student/advisor hợp lệ.")
if len(edge_rows) > MAX_RECORDS_TOTAL:
    raise ValueError(f"Raw records vượt cap {MAX_RECORDS_TOTAL}: {len(edge_rows)}")

edges_raw = pd.DataFrame(edge_rows)
raw_record_count = len(edges_raw)
edges_raw = edges_raw.drop_duplicates(subset=["person", "object", "relationship_label"])
entity_ids = sorted(set(edges_raw["person"]) | set(edges_raw["object"]))
print("Một file raw từ một request UNION:", raw_file.name)
print("Records theo relation:", records_by_relation)
print(f"Raw records: {raw_record_count:,} (cap {MAX_RECORDS_TOTAL:,})")
print(f"Unique directed edges: {len(edges_raw):,} | Entities: {len(entity_ids):,}")
print("Relation labels giữ riêng:", sorted(edges_raw["relationship_label"].unique()))
edges_raw.head()

In [ ]:
from SPARQLWrapper import SPARQLWrapper, JSON as SPARQL_JSON

METADATA_CACHE_PATH = RAW_PARQUET_DIR / "wikidata_node_metadata.parquet"
metadata_columns = [
    "id", "name", "description", "instance_of", "occupation",
    "sex_or_gender", "birth_year", "birth_place", "country",
]
if METADATA_CACHE_PATH.exists():
    metadata_cache = pd.read_parquet(METADATA_CACHE_PATH)
    metadata_cache = metadata_cache.reindex(columns=metadata_columns)
else:
    metadata_cache = pd.DataFrame(columns=metadata_columns)

cached_ids = set(metadata_cache["id"].dropna().astype(str))
ids_to_fetch = [entity_id for entity_id in entity_ids if entity_id not in cached_ids]
print(f"Metadata cache: {len(cached_ids):,} IDs | cần truy vấn mới: {len(ids_to_fetch):,}")


def fetch_entity_metadata(entity_batch, user_agent, max_retries=5):
    values = " ".join(f"wd:{entity_id}" for entity_id in entity_batch)
    query = f"""
    SELECT ?entity ?entityLabel ?entityDescription ?instanceOfLabel ?occupationLabel
           ?genderLabel ?birthDate ?birthPlaceLabel ?countryLabel
    WHERE {{
      VALUES ?entity {{ {values} }}
      OPTIONAL {{ ?entity wdt:P31 ?instanceOf. }}
      OPTIONAL {{ ?entity wdt:P106 ?occupation. }}
      OPTIONAL {{ ?entity wdt:P21 ?gender. }}
      OPTIONAL {{ ?entity wdt:P569 ?birthDate. }}
      OPTIONAL {{ ?entity wdt:P19 ?birthPlace. }}
      OPTIONAL {{ ?entity wdt:P27 ?country. }}
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en,vi". }}
    }}
    """
    client = SPARQLWrapper("https://query.wikidata.org/sparql", agent=user_agent)
    client.setReturnFormat(SPARQL_JSON)
    client.setTimeout(300)
    client.setQuery(query)
    for attempt in range(max_retries):
        try:
            return client.query().convert()["results"]["bindings"]
        except Exception:
            if attempt + 1 == max_retries:
                raise
            time.sleep(min(5 * (attempt + 1), 30))


metadata_rows = []
BATCH_SIZE_METADATA = 50
for batch_start in range(0, len(ids_to_fetch), BATCH_SIZE_METADATA):
    batch = ids_to_fetch[batch_start:batch_start + BATCH_SIZE_METADATA]
    try:
        bindings = fetch_entity_metadata(batch, USER_AGENT)
    except Exception as error:
        raise RuntimeError(
            f"Truy vấn metadata lỗi tại batch bắt đầu {batch_start}; dữ liệu đã checkpoint vẫn còn ở {METADATA_CACHE_PATH}."
        ) from error

    for binding in bindings:
        entity_id = normalize_qid(binding_value(binding, "entity"))
        if entity_id:
            metadata_rows.append({
                "id": entity_id,
                "name": binding_value(binding, "entityLabel"),
                "description": binding_value(binding, "entityDescription"),
                "instance_of": binding_value(binding, "instanceOfLabel"),
                "occupation": binding_value(binding, "occupationLabel"),
                "sex_or_gender": binding_value(binding, "genderLabel"),
                "birth_year": binding_value(binding, "birthDate"),
                "birth_place": binding_value(binding, "birthPlaceLabel"),
                "country": binding_value(binding, "countryLabel"),
            })

    if metadata_rows:
        metadata_cache = pd.concat([metadata_cache, pd.DataFrame(metadata_rows)], ignore_index=True)
        metadata_rows.clear()
        metadata_cache = metadata_cache.drop_duplicates("id", keep="last")
        metadata_cache.to_parquet(METADATA_CACHE_PATH, index=False)
    if batch_start + BATCH_SIZE_METADATA < len(ids_to_fetch):
        time.sleep(1.0)

metadata_cache = metadata_cache.drop_duplicates("id", keep="last")
metadata_cache.to_parquet(METADATA_CACHE_PATH, index=False)
print(f"Đã checkpoint metadata: {len(metadata_cache):,} entities -> {METADATA_CACHE_PATH}")

## 4. Tiền Xử Lý Dữ Liệu
Chuẩn hóa ID QID, loại bản ghi trùng, ghép thuộc tính người/vật thể với metadata, gán node type và local index theo từng loại. Schema đầu ra khớp với `GraphDataProcessor` trong dự án.

In [ ]:
def first_non_empty(series):
    values = series.dropna().astype(str).str.strip()
    values = values[values.ne("") & values.ne("nan")]
    return values.iloc[0] if not values.empty else None


def join_unique(series):
    values = series.dropna().astype(str).str.strip()
    values = values[values.ne("") & values.ne("nan")]
    return ", ".join(pd.unique(values)) if not values.empty else ""


def year_value(value):
    if value is None or pd.isna(value):
        return np.nan
    match = re.search(r"(?:^|[-+])(\d{4})", str(value))
    return float(match.group(1)) if match else pd.to_numeric(value, errors="coerce")


def normalize_node_type(value):
    if not value:
        return "entity"
    return re.sub(r"[^a-z0-9]+", "_", str(value).strip().lower()).strip("_") or "entity"

person_attrs = edges_raw.groupby("person", as_index=False).agg(
    name=("person_label", first_non_empty),
    description=("person_description", first_non_empty),
    birth_year=("birth_year", first_non_empty),
    birth_place=("birth_place", first_non_empty),
    country=("country", first_non_empty),
).rename(columns={"person": "id"})
object_attrs = edges_raw.groupby("object", as_index=False).agg(
    name=("object_label", first_non_empty),
    description=("object_description", first_non_empty),
).rename(columns={"object": "id"})

metadata = metadata_cache.copy()
metadata["id"] = metadata["id"].astype(str)
metadata = metadata.drop_duplicates("id", keep="last")
metadata = metadata.rename(columns={
    "instance_of": "instance_type",
    "sex_or_gender": "sex_or_gender_metadata",
    "birth_year": "birth_year_metadata",
    "birth_place": "birth_place_metadata",
    "country": "country_metadata",
    "occupation": "occupation_metadata",
    "name": "metadata_name",
    "description": "metadata_description",
})

nodes = pd.DataFrame({"id": entity_ids}).merge(metadata, on="id", how="left")
nodes = nodes.merge(person_attrs, on="id", how="left", suffixes=("", "_person"))
nodes = nodes.merge(object_attrs, on="id", how="left", suffixes=("", "_object"))

for target, choices in {
    "name": ["name", "name_person", "name_object", "metadata_name"],
    "description": ["description", "description_person", "description_object", "metadata_description"],
    "birth_year": ["birth_year", "birth_year_metadata"],
    "birth_place": ["birth_place", "birth_place_metadata"],
    "country": ["country", "country_metadata"],
    "occupation": ["occupation_metadata"],
    "sex_or_gender": ["sex_or_gender_metadata"],
}.items():
    present = [column for column in choices if column in nodes.columns]
    nodes[target] = nodes[present].bfill(axis=1).iloc[:, 0] if present else None

nodes["birth_year"] = nodes["birth_year"].map(year_value)
nodes["name"] = nodes["name"].fillna(nodes["id"])
nodes["description"] = nodes["description"].fillna("")
nodes["occupation"] = nodes["occupation"].fillna("")
nodes["country"] = nodes["country"].fillna("")
nodes["birth_place"] = nodes["birth_place"].fillna("")
nodes["sex_or_gender"] = nodes["sex_or_gender"].fillna("")
nodes["interests"] = ""

person_ids = set(edges_raw["person"])
nodes["type"] = nodes["instance_type"].map(normalize_node_type)
nodes.loc[nodes["id"].isin(person_ids), "type"] = "human"
nodes["pyg_id"] = nodes.groupby("type", sort=False).cumcount().astype("int64")

node_types = nodes.set_index("id")["type"]
edges = edges_raw[["person", "object", "relationship_label"]].copy()
edges["person_type"] = edges["person"].map(node_types)
edges["object_type"] = edges["object"].map(node_types)
edges = edges.dropna(subset=["person_type", "object_type"])
edges = edges.drop_duplicates(["person", "relationship_label", "object"]).reset_index(drop=True)

required_node_columns = [
    "id", "name", "description", "occupation", "country", "birth_place",
    "sub_type", "sex_or_gender", "birth_year", "interests", "type", "pyg_id",
]
nodes["sub_type"] = nodes["instance_type"].fillna("")
missing_columns = set(required_node_columns) - set(nodes.columns)
if missing_columns:
    raise ValueError(f"Thiếu cột node theo model: {sorted(missing_columns)}")
nodes = nodes[required_node_columns]

print(f"Nodes: {len(nodes):,} | edges: {len(edges):,}")
print("Node types:", nodes["type"].nunique(), "| Relations:", edges["relationship_label"].nunique())
print("Feature columns required:", required_node_columns)
nodes.head()

In [ ]:
# DataFrame nodes giữ các cột feature; ghép lại instance_type để gán type đã chuẩn hóa trước khi build graph.
nodes = nodes.merge(
    metadata[["id", "instance_type"]].drop_duplicates("id"),
    on="id",
    how="left",
)

# Dùng các quan hệ thuộc tính đã thu thập (ví dụ sport) làm trường interests cho embedding văn bản.
interest_rows = edges_raw[
    edges_raw["relationship_label"].isin({"sport", "field_of_work", "instrument", "genre"})
]
if not interest_rows.empty:
    interests_by_person = interest_rows.groupby("person")["object_label"].agg(join_unique)
    nodes["interests"] = nodes["id"].map(interests_by_person).fillna("")

assert nodes["id"].is_unique, "Mỗi QID cần ánh xạ đúng một node type/local id."
assert set(edges["person"]).issubset(set(nodes["id"]))
assert set(edges["object"]).issubset(set(nodes["id"]))
print("Missing names:", int(nodes["name"].isna().sum()))
print("Interest values populated:", int(nodes["interests"].astype(bool).sum()))

In [ ]:
# Dùng transformer/repository hiện có để tạo graph mà các tab phân tích của ứng dụng đọc.
from config.settings import GRAPH_PATH
from infrastructure.repositories import PickleGraphRepository

graph_transformer = GraphTransformer()
relationship_graph = graph_transformer.build_graph(edges, nodes)
graph_repository = PickleGraphRepository(GRAPH_PATH)
if not graph_repository.save_graph(relationship_graph):
    raise IOError(f"Không lưu được relationship graph: {GRAPH_PATH}")

# Load lại file đã lưu rồi đối soát như trong Social Link Analysis and Prediction.ipynb.
relationship_graph = graph_repository.load_graph()
if relationship_graph is None:
    raise IOError(f"Không load lại được relationship graph: {GRAPH_PATH}")
assert relationship_graph.vcount() == nodes["id"].nunique()
assert relationship_graph.ecount() == len(edges)
assert relationship_graph.vs["name"] == nodes["id"].tolist()
assert relationship_graph.vs["label"] == nodes["name"].tolist()
assert relationship_graph.ecount() > 0, "Graph không có cạnh sau bước tiền xử lý."
print("Graph save/load integrity test PASS")
print("Saved relationship graph:", GRAPH_PATH)
print("Graph nodes/edges:", relationship_graph.vcount(), relationship_graph.ecount())

## 5. Lưu Dữ Liệu Đã Thu Thập
Lưu bảng node/cạnh, metadata, PyG graph, mapping adjacency và checkpoint model để tái sử dụng. Dữ liệu lớn được ghi Parquet/PyTorch; model không tự tải dữ liệu lại khi đã có checkpoint.

In [ ]:
from config.settings import METADATA_PATH
METADATA_PATH.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
# Giữ edge type nhỏ trong graph kiểm thử; khôi phục ngưỡng cấu hình ngay sau khi build.
import core.ai.data_processor as data_processor_module

original_min_edge_count = data_processor_module.MIN_EDGE_COUNT
if TEST_MODE:
    data_processor_module.MIN_EDGE_COUNT = 1

In [ ]:
# Xử lý kiểu thiếu sau khi đã ghép metadata.
nodes["type"] = nodes["instance_type"].map(
    lambda value: "entity" if pd.isna(value) or not str(value).strip() else normalize_node_type(value)
)
nodes.loc[nodes["id"].isin(person_ids), "type"] = "human"
nodes["pyg_id"] = nodes.groupby("type", sort=False).cumcount().astype("int64")
node_types = nodes.set_index("id")["type"]
edges["person_type"] = edges["person"].map(node_types)
edges["object_type"] = edges["object"].map(node_types)
edges = edges.dropna(subset=["person_type", "object_type"]).copy()

RAW_PARQUET_DIR.mkdir(parents=True, exist_ok=True)
CLEAN_DATA_DIR.mkdir(parents=True, exist_ok=True)
nodes.to_parquet(CLEAN_DATA_DIR / "nodes_data.parquet", index=False)
edges.to_parquet(CLEAN_DATA_DIR / "edges_data.parquet", index=False)

from config.settings import ADJACENCY_PATH, METADATA_PATH
from core.ai.data_processor import GraphDataProcessor
from infrastructure.repositories import PyGDataRepository

processor = GraphDataProcessor()
pyg_data, adjacency = processor.process_graph_to_pyg(
    df_edges=edges.copy(),
    df_nodes=nodes.copy(),
)
feature_dims = {node_type: int(pyg_data[node_type].x.size(1)) for node_type in pyg_data.node_types}
assert len(set(feature_dims.values())) == 1, f"Node feature dims không đồng nhất: {feature_dims}"
actual_feature_dim = next(iter(feature_dims.values()))
expected_feature_dim = 384 + 2 + edges["relationship_label"].nunique() + 1
assert actual_feature_dim == expected_feature_dim, (
    f"Feature dimension mismatch: actual={actual_feature_dim}, expected={expected_feature_dim}"
)

repository = PyGDataRepository(PYG_DATA_PATH, ADJACENCY_PATH)
repository.save_data(pyg_data)
repository.save_adjacency(adjacency)
with open(METADATA_PATH, "wb") as metadata_file:
    import pickle
    pickle.dump(pyg_data.metadata(), metadata_file)

print("Saved nodes:", CLEAN_DATA_DIR / "nodes_data.parquet")
print("Saved edges:", CLEAN_DATA_DIR / "edges_data.parquet")
print("Saved PyG graph:", PYG_DATA_PATH)
print("Feature dimensions by node type:", feature_dims)
print("Actual feature width:", actual_feature_dim)

In [ ]:
data_processor_module.MIN_EDGE_COUNT = original_min_edge_count
print("Restored project MIN_EDGE_COUNT:", original_min_edge_count)

In [ ]:
# Sanity check dữ liệu HeteroData trước khi cho trainer đọc.
import torch

assert pyg_data.num_nodes == len(nodes), (pyg_data.num_nodes, len(nodes))
assert pyg_data.num_edges > 0, "PyG graph không có cạnh sau tiền xử lý."
assert set(nodes["type"].unique()).issubset(set(pyg_data.node_types))
for edge_type in pyg_data.edge_types:
    edge_index = pyg_data[edge_type].edge_index
    src_type, _, dst_type = edge_type
    assert edge_index.dtype == torch.long
    assert edge_index.dim() == 2 and edge_index.size(0) == 2
    assert edge_index.numel() > 0
    assert int(edge_index[0].min()) >= 0
    assert int(edge_index[0].max()) < pyg_data[src_type].num_nodes
    assert int(edge_index[1].min()) >= 0
    assert int(edge_index[1].max()) < pyg_data[dst_type].num_nodes

print("PyG graph integrity test PASS")
print("Node types:", len(pyg_data.node_types), "| Edge types incl. reverse:", len(pyg_data.edge_types))
print("Nodes:", pyg_data.num_nodes, "| Edges:", pyg_data.num_edges)
print("Feature widths:", feature_dims)

### Chia tập và train GraphSAGE theo README
Tách cạnh cùng cạnh đảo chiều để hạn chế leakage. Encoder/decoder lấy trực tiếp từ `core/ai/gnn_architecture.py`; loss dùng logits với `BCEWithLogitsLoss`, sigmoid chỉ dùng khi tính xác suất/AUC.

In [ ]:
# Gọi trực tiếp trainer của dự án, dùng LinkPredictionModel trong core/ai/gnn_architecture.py.
# TEST_MODE=True sẽ train trên lát cắt nhỏ; TEST_MODE=False sẽ train trên dữ liệu đầy đủ.
RUN_TRAINING = True

if RUN_TRAINING:
    from scripts.train_model import run_optimization
    run_optimization(data=pyg_data)
else:
    print("Đã bỏ qua training. Đặt RUN_TRAINING=True để gọi scripts.train_model.run_optimization().")

## 6. Đánh Giá Model
Trainer ghi loss từng epoch, ROC-AUC và Average Precision trên validation, sau đó đánh giá lại checkpoint tốt nhất trên test set. Các metric này dùng nhãn âm/dương thật từ `RandomLinkSplit`.

In [ ]:
from pathlib import Path
import json
import pandas as pd

history_path = Path(TRAINING_HISTORY_PATH)
if not history_path.is_file():
    raise FileNotFoundError(
        f"Chưa có training history tại {history_path}. Hãy chạy cell train trước và kiểm tra log trainer."
    )

with history_path.open("r", encoding="utf-8") as history_file:
    training_history = json.load(history_file)

history_frame = pd.DataFrame({
    "epoch": training_history.get("epoch", []),
    "loss": training_history.get("loss", []),
    "val_roc_auc": training_history.get("val_auc", []),
    "val_average_precision": training_history.get("val_ap", []),
})
if history_frame.empty:
    raise ValueError("Training history rỗng; chưa có epoch nào hoàn thành.")

valid_val_auc = history_frame.dropna(subset=["val_roc_auc"])
if not valid_val_auc.empty:
    best_row = valid_val_auc.loc[valid_val_auc["val_roc_auc"].idxmax()]
    best_epoch = int(best_row["epoch"])
    best_val_auc = float(best_row["val_roc_auc"])
    best_val_ap = best_row["val_average_precision"]
else:
    best_epoch = None
    best_val_auc = None
    best_val_ap = None

final_evaluation = pd.DataFrame([{
    "best_epoch": best_epoch,
    "best_val_roc_auc": best_val_auc,
    "best_val_average_precision": best_val_ap,
    "test_roc_auc": training_history.get("test_auc"),
    "test_average_precision": training_history.get("test_ap"),
    "student_of_edges": records_by_query.get("student", 0),
    "advisor_of_edges": records_by_query.get("advisor", 0),
    "unique_edges_after_dedup": len(edges),
}])

print("Đánh giá trên tập test độc lập:")
display(final_evaluation)
print("Lịch sử huấn luyện:")
display(history_frame)
print("Ghi chú: ROC-AUC 0.5 tương đương xếp hạng ngẫu nhiên; AP cần so với tỷ lệ positive của tập test.")